# 03 - Feature engineering from FEN

A raw FEN string isn't something logistic regression can use directly — this notebook turns each FEN into a numeric feature vector using `python-chess`.

Feature set (see [`src/features.py`](../src/features.py)):
- material counts per piece type per side, and `material_diff`
- side to move, castling rights (4 flags)
- mobility (legal move count) per side, and `mobility_diff`
- center-square (d4/d5/e4/e5) control per side, and the diff
- whether the side to move is in check
- move number

Runs on the already-split `data/processed/train.csv` / `test.csv` from the labels notebook — features are computed *after* the split so there's no risk of the split logic ever depending on engineered features.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.features import extract_features, add_features

train_df = pd.read_csv('../data/processed/train.csv', low_memory=False)
test_df = pd.read_csv('../data/processed/test.csv', low_memory=False)

train_df.shape, test_df.shape

((822936, 19), (208378, 19))

## Sanity-check on a handful of positions
Before running this over 1M+ rows, confirm the features look right on a few known positions (starting position, and the row-1 position after `e2e4`).

In [2]:
starting_fen = 'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1'
after_e4_fen = train_df.loc[train_df['fen'].str.startswith('rnbqkbnr/pppppppp/8/8/4P3'), 'fen'].iloc[0]

for label, fen in [('starting position', starting_fen), ('after 1. e4', after_e4_fen)]:
    print(label)
    for k, v in extract_features(fen).items():
        print(f'  {k}: {v}')
    print()

starting position
  white_pawns: 8
  white_knights: 2
  white_bishops: 2
  white_rooks: 2
  white_queens: 1
  black_pawns: 8
  black_knights: 2
  black_bishops: 2
  black_rooks: 2
  black_queens: 1
  material_diff: 0
  side_to_move_white: 1
  white_kingside_castle: 1
  white_queenside_castle: 1
  black_kingside_castle: 1
  black_queenside_castle: 1
  mobility_white: 20
  mobility_black: 20
  mobility_diff: 0
  white_center_control: 0
  black_center_control: 0
  center_control_diff: 0
  in_check: 0
  fullmove_number: 1

after 1. e4
  white_pawns: 8
  white_knights: 2
  white_bishops: 2
  white_rooks: 2
  white_queens: 1
  black_pawns: 8
  black_knights: 2
  black_bishops: 2
  black_rooks: 2
  black_queens: 1
  material_diff: 0
  side_to_move_white: 0
  white_kingside_castle: 1
  white_queenside_castle: 1
  black_kingside_castle: 1
  black_queenside_castle: 1
  mobility_white: 30
  mobility_black: 20
  mobility_diff: 10
  white_center_control: 1
  black_center_control: 0
  center_control

## Extract features for train and test
Benchmarked at ~0.13ms/row on a 5,000-row sample — full train (~823K rows) and test (~208K rows) should take a couple of minutes combined.

In [3]:
%%time
train_df = add_features(train_df)
test_df = add_features(test_df)

train_df.shape, test_df.shape

CPU times: user 2min 23s, sys: 1.34 s, total: 2min 24s
Wall time: 2min 24s


((822936, 43), (208378, 43))

## Quick look at the engineered features
Check summary stats — do `material_diff` and `mobility_diff` correlate with `white_win` in the direction we'd expect?

In [4]:
feature_cols = [
    'material_diff', 'mobility_diff', 'center_control_diff',
    'side_to_move_white', 'in_check', 'fullmove_number',
]
print(train_df[feature_cols + ['white_win']].corr()['white_win'])
train_df[feature_cols].describe()

material_diff          0.350671
mobility_diff          0.254958
center_control_diff    0.130438
side_to_move_white    -0.007133
in_check               0.000043
fullmove_number       -0.013220
white_win              1.000000
Name: white_win, dtype: float64


,material_diff,mobility_diff,center_control_diff,side_to_move_white,in_check,fullmove_number
count,822936.000000,822936.000000,822936.000000,822936.000000,822936.000000,822936.000000
mean,0.028127,1.381394,0.120422,0.495915,0.077033,20.808069
std,4.267475,14.134265,1.339517,0.499984,0.266644,14.815164
min,-40.000000,-83.000000,-4.000000,0.000000,0.000000,1.000000
25%,-1.000000,-6.000000,-1.000000,0.000000,0.000000,9.000000
50%,0.000000,2.000000,0.000000,0.000000,0.000000,18.000000
75%,1.000000,9.000000,1.000000,1.000000,0.000000,29.000000
max,40.000000,88.000000,4.000000,1.000000,1.000000,117.000000


## Save feature-enriched splits

In [5]:
train_df.to_csv('../data/processed/train_features.csv', index=False)
test_df.to_csv('../data/processed/test_features.csv', index=False)
print('saved.')

saved.
